# Module 14: End-to-End Scenario — Credit Scoring (Predictive AI)

## What You'll Learn

- Complete predictive AI lifecycle on RHOAI with Feast
- Based on Scenario A from the data strategy proposal
- Steps: Define data sources → Feature engineering → Materialize → Train model → Online serving → Lineage tracking
- Uses concepts from Modules 01–07
- Demonstrates the full value of Feast for the predictive AI workload pattern

## Prerequisites

- Modules 01–05 completed (core concepts, stores, feature engineering)
- Feast deployed via operator (Module 12) or local `feature_store.yaml`
- `feast`, `pandas`, `scikit-learn`, `xgboost` installed

---

> **🗺️ DATA STRATEGY**: This is **Scenario A** from the data-strategy-proposal. Exercises **Pillar 2** (compute), **Pillar 3** (Feast for predictive AI), and **Pillar 4** (lineage). The canonical example of Feast doing what it was designed for.
>
> **📍 RHOAI STATUS**: All core Feast features used here are GA downstream. Gaps affect observability and compute orchestration — noted at each step.

## Scenario Overview

A bank wants to score credit applications in real time. The workflow:

```
┌─────────────┐    ┌──────────────┐    ┌─────────────┐    ┌──────────────┐
│ Data Sources│───▶│ Feature Eng. │───▶│ Materialize │───▶│ Online Store │
│ (3 sources) │    │ (Feast FVs)  │    │ (batch)     │    │ (serving)    │
└─────────────┘    └──────────────┘    └─────────────┘    └──────────────┘
                          │                                        │
                          ▼                                        ▼
                   ┌──────────────┐                        ┌──────────────┐
                   │ Train Model  │                        │ Score Apps   │
                   │ (offline)    │                        │ (real-time)  │
                   └──────────────┘                        └──────────────┘
```

### Data Sources

1. **Applications** — applicant demographics, requested amount
2. **Payment History** — monthly payment behavior per customer
3. **Credit Bureau** — external credit scores and tradelines

### Known Gaps in This Scenario

| Gap | Impact on Scenario |
|-----|-------------------|
| No auto-alerting on stale features | Must monitor freshness manually (Module 13) |
| Ray/Spark not in CRD | Large-scale materialization needs manual compute config |
| Lineage breaks at training boundary | OpenLineage tracks features, not model consumption |
| No model→feature link in any UI | Must log feature metadata to MLflow manually |

## Step 1: Generate Realistic Credit Scoring Data

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
os.makedirs("data", exist_ok=True)

N_CUSTOMERS = 500
N_APPLICATIONS = 2000

# --- Credit Bureau Data (external, updated monthly) ---
bureau_records = []
for cid in range(1, N_CUSTOMERS + 1):
    for month in range(24):
        ts = datetime(2023, 1, 1) + timedelta(days=30 * month)
        bureau_records.append({
            "customer_id": cid,
            "event_timestamp": ts,
            "bureau_score": np.random.randint(300, 850),
            "num_open_accounts": np.random.randint(1, 15),
            "total_credit_limit": np.random.randint(5000, 100000),
            "delinquencies_12m": np.random.randint(0, 5),
            "bankruptcies": np.random.choice([0, 0, 0, 0, 1]),
        })

bureau_df = pd.DataFrame(bureau_records)
bureau_df.to_parquet("data/credit_bureau.parquet")
print(f"Credit bureau: {len(bureau_df)} records")

# --- Payment History (monthly aggregates) ---
payment_records = []
for cid in range(1, N_CUSTOMERS + 1):
    for month in range(24):
        ts = datetime(2023, 1, 1) + timedelta(days=30 * month)
        payment_records.append({
            "customer_id": cid,
            "event_timestamp": ts,
            "on_time_payments": np.random.randint(0, 5),
            "late_payments": np.random.randint(0, 3),
            "missed_payments": np.random.randint(0, 2),
            "avg_payment_amount": round(np.random.uniform(100, 2000), 2),
            "utilization_ratio": round(np.random.uniform(0.1, 0.95), 3),
        })

payment_df = pd.DataFrame(payment_records)
payment_df.to_parquet("data/payment_history.parquet")
print(f"Payment history: {len(payment_df)} records")

# --- Loan Applications ---
app_records = []
for i in range(N_APPLICATIONS):
    cid = np.random.randint(1, N_CUSTOMERS + 1)
    ts = datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 365))
    income = np.random.randint(25000, 200000)
    amount = np.random.randint(1000, 50000)
    app_records.append({
        "application_id": i + 1,
        "customer_id": cid,
        "event_timestamp": ts,
        "annual_income": income,
        "requested_amount": amount,
        "employment_years": round(np.random.uniform(0.5, 30), 1),
        "debt_to_income": round(amount / max(income, 1), 3),
        "defaulted": np.random.choice([0, 1], p=[0.85, 0.15]),  # label
    })

apps_df = pd.DataFrame(app_records)
apps_df.to_parquet("data/applications.parquet")
print(f"Applications: {len(apps_df)} records, default rate: {apps_df['defaulted'].mean():.1%}")

## Step 2: Define Entities, Feature Views, and On-Demand Transformations

In [ ]:
from feast import Entity, FeatureView, Field, FileSource, FeatureService
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Float64, Int64

# --- Entities ---
customer = Entity(name="customer", join_keys=["customer_id"])
application = Entity(name="application", join_keys=["application_id"])

# --- Data Sources ---
bureau_source = FileSource(
    name="credit_bureau_source",
    path=os.path.abspath("data/credit_bureau.parquet"),
    timestamp_field="event_timestamp",
)

payment_source = FileSource(
    name="payment_history_source",
    path=os.path.abspath("data/payment_history.parquet"),
    timestamp_field="event_timestamp",
)

apps_source = FileSource(
    name="applications_source",
    path=os.path.abspath("data/applications.parquet"),
    timestamp_field="event_timestamp",
)

# --- Feature Views ---
bureau_fv = FeatureView(
    name="credit_bureau_features",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="bureau_score", dtype=Int64),
        Field(name="num_open_accounts", dtype=Int64),
        Field(name="total_credit_limit", dtype=Int64),
        Field(name="delinquencies_12m", dtype=Int64),
        Field(name="bankruptcies", dtype=Int64),
    ],
    source=bureau_source,
)

payment_fv = FeatureView(
    name="payment_history_features",
    entities=[customer],
    ttl=timedelta(days=30),
    schema=[
        Field(name="on_time_payments", dtype=Int64),
        Field(name="late_payments", dtype=Int64),
        Field(name="missed_payments", dtype=Int64),
        Field(name="avg_payment_amount", dtype=Float64),
        Field(name="utilization_ratio", dtype=Float32),
    ],
    source=payment_source,
)

application_fv = FeatureView(
    name="application_features",
    entities=[application],
    ttl=timedelta(days=7),
    schema=[
        Field(name="customer_id", dtype=Int64),
        Field(name="annual_income", dtype=Int64),
        Field(name="requested_amount", dtype=Int64),
        Field(name="employment_years", dtype=Float32),
        Field(name="debt_to_income", dtype=Float32),
        Field(name="defaulted", dtype=Int64),
    ],
    source=apps_source,
)

print("Defined 2 entities, 3 feature views")

In [ ]:
# --- On-Demand Feature View: risk score from bureau + payment features ---
@on_demand_feature_view(
    sources=[bureau_fv, payment_fv],
    schema=[
        Field(name="composite_risk_score", dtype=Float32),
        Field(name="payment_reliability", dtype=Float32),
    ],
)
def credit_risk_odfv(inputs: pd.DataFrame) -> pd.DataFrame:
    """Compute composite risk score at request time."""
    df = pd.DataFrame()
    bureau = inputs["bureau_score"].fillna(500)
    delinq = inputs["delinquencies_12m"].fillna(0)
    missed = inputs["missed_payments"].fillna(0)
    on_time = inputs["on_time_payments"].fillna(0)
    late = inputs["late_payments"].fillna(0)

    df["composite_risk_score"] = (
        (850 - bureau) / 550 * 0.5
        + delinq * 0.1
        + missed * 0.15
    ).clip(0, 1).astype(np.float32)

    total_payments = on_time + late + missed
    df["payment_reliability"] = np.where(
        total_payments > 0,
        on_time / total_payments,
        0.5
    ).astype(np.float32)

    return df

# --- Feature Service: group features for credit scoring ---
credit_scoring_service = FeatureService(
    name="credit_scoring_v1",
    features=[
        bureau_fv,
        payment_fv,
        credit_risk_odfv,
    ],
)

print("Defined ODFV: credit_risk_odfv")
print("Defined FeatureService: credit_scoring_v1")

## Step 3: Apply and Materialize

In [ ]:
import yaml

# Create feature_store.yaml for local execution
feature_store_config = {
    "project": "credit_scoring",
    "provider": "local",
    "registry": "data/registry.db",
    "offline_store": {"type": "duckdb"},
    "online_store": {"type": "sqlite", "path": "data/online_store.db"},
}

os.makedirs("feature_repo", exist_ok=True)
with open("feature_repo/feature_store.yaml", "w") as f:
    yaml.dump(feature_store_config, f)

# Write feature definitions to feature_repo
feature_defs = '''
from datetime import timedelta
import os
import pandas as pd
import numpy as np
from feast import Entity, FeatureView, Field, FileSource, FeatureService
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Float64, Int64

customer = Entity(name="customer", join_keys=["customer_id"])
application = Entity(name="application", join_keys=["application_id"])

bureau_source = FileSource(name="credit_bureau_source",
    path=os.path.abspath("data/credit_bureau.parquet"), timestamp_field="event_timestamp")
payment_source = FileSource(name="payment_history_source",
    path=os.path.abspath("data/payment_history.parquet"), timestamp_field="event_timestamp")
apps_source = FileSource(name="applications_source",
    path=os.path.abspath("data/applications.parquet"), timestamp_field="event_timestamp")

bureau_fv = FeatureView(name="credit_bureau_features", entities=[customer],
    ttl=timedelta(days=30), schema=[
        Field(name="bureau_score", dtype=Int64), Field(name="num_open_accounts", dtype=Int64),
        Field(name="total_credit_limit", dtype=Int64), Field(name="delinquencies_12m", dtype=Int64),
        Field(name="bankruptcies", dtype=Int64)], source=bureau_source)

payment_fv = FeatureView(name="payment_history_features", entities=[customer],
    ttl=timedelta(days=30), schema=[
        Field(name="on_time_payments", dtype=Int64), Field(name="late_payments", dtype=Int64),
        Field(name="missed_payments", dtype=Int64), Field(name="avg_payment_amount", dtype=Float64),
        Field(name="utilization_ratio", dtype=Float32)], source=payment_source)

application_fv = FeatureView(name="application_features", entities=[application],
    ttl=timedelta(days=7), schema=[
        Field(name="customer_id", dtype=Int64), Field(name="annual_income", dtype=Int64),
        Field(name="requested_amount", dtype=Int64), Field(name="employment_years", dtype=Float32),
        Field(name="debt_to_income", dtype=Float32), Field(name="defaulted", dtype=Int64)],
    source=apps_source)
'''

with open("feature_repo/features.py", "w") as f:
    f.write(feature_defs)

print("Created feature_repo/ with feature_store.yaml and features.py")

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")
store.apply([customer, application, bureau_fv, payment_fv, application_fv, credit_risk_odfv, credit_scoring_service])
print("feast apply complete — entities and feature views registered")

# Materialize features to online store
end_date = datetime.now()
start_date = end_date - timedelta(days=365)
store.materialize(start_date, end_date)
print(f"Materialized {start_date.date()} to {end_date.date()}")

# On RHOAI with operator: materialization runs via CronJob or KFP pipeline
# > **⚠️ GAP**: Operator uses oc exec for materialization — not production-grade for large datasets

## Step 4: Train a Gradient Boosted Model Using Feast Features

In [ ]:
# Retrieve historical features for training (point-in-time correct)
entity_df = apps_df[["application_id", "customer_id", "event_timestamp", "defaulted"]].copy()
entity_df = entity_df.rename(columns={"event_timestamp": "event_timestamp"})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        bureau_fv[["bureau_score", "delinquencies_12m", "bankruptcies"]],
        payment_fv[["on_time_payments", "late_payments", "missed_payments", "utilization_ratio"]],
        application_fv[["annual_income", "requested_amount", "debt_to_income", "employment_years"]],
    ],
).to_df()

print(f"Training dataset: {training_df.shape}")
print(f"Columns: {list(training_df.columns)}")
training_df.head(3)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb

feature_cols = [
    "bureau_score", "delinquencies_12m", "bankruptcies",
    "on_time_payments", "late_payments", "missed_payments", "utilization_ratio",
    "annual_income", "requested_amount", "debt_to_income", "employment_years",
]

X = training_df[feature_cols].fillna(0)
y = training_df["defaulted"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    eval_metric="auc", random_state=42
)
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC: {auc:.4f}")
print(classification_report(y_test, model.predict(X_test), target_names=["no_default", "default"]))

# > **⚠️ GAP**: Lineage breaks here — no automatic link from Feast features to trained model
# Log feature metadata to MLflow manually for traceability

## Step 5: Serve Predictions Using Online Features

In [ ]:
# Simulate real-time scoring for a new application
new_application = {
    "application_id": 9999,
    "customer_id": 42,
}

# Fetch online features (same definitions as training — no skew)
online_features = store.get_online_features(
    features=[
        bureau_fv[["bureau_score", "delinquencies_12m", "bankruptcies"]],
        payment_fv[["on_time_payments", "late_payments", "missed_payments", "utilization_ratio"]],
    ],
    entity_rows=[{"customer_id": new_application["customer_id"]}],
).to_dict()

print("Online features retrieved:")
for key, values in online_features.items():
    if key != "customer_id":
        print(f"  {key}: {values[0]}")

# Build feature vector and score
score_features = pd.DataFrame({
    col: [online_features[col][0] if online_features[col][0] is not None else 0]
    for col in ["bureau_score", "delinquencies_12m", "bankruptcies",
                "on_time_payments", "late_payments", "missed_payments", "utilization_ratio"]
})
# Add application-time features (provided at request)
score_features["annual_income"] = 75000
score_features["requested_amount"] = 15000
score_features["debt_to_income"] = 0.2
score_features["employment_years"] = 5.0

default_probability = model.predict_proba(score_features[feature_cols])[0, 1]
decision = "APPROVE" if default_probability < 0.3 else "REVIEW" if default_probability < 0.6 else "DENY"

print(f"\nApplication {new_application['application_id']} (customer {new_application['customer_id']}):")
print(f"  Default probability: {default_probability:.4f}")
print(f"  Decision: {decision}")

## Step 6 (Optional): Emit OpenLineage Events

If OpenLineage is configured (Module 07), `feast apply` and `feast materialize` automatically emit lineage events. On RHOAI, this requires a ConfigMap overlay (Module 12) since the CRD does not expose lineage config.

In [ ]:
# OpenLineage is enabled via feature_store.yaml overlay:
lineage_config = """
lineage:
  enabled: true
  backend_url: http://marquez.my-ds-project.svc:5000

# When enabled, feast apply emits:
#   - Dataset: credit_bureau_features (schema)
#   - Dataset: payment_history_features (schema)
#   - Job: feast_apply_credit_scoring

# feast materialize emits:
#   - Job: feast_materialize_credit_bureau_features
#   - Input: credit_bureau_source → Output: online store
"""
print(lineage_config)
print("View lineage in Marquez UI (Module 13)")
print("⚠️ GAP: Lineage stops at feature store — model training not tracked")

## Step 7: KFP Pipeline Orchestration

For production, wrap this workflow in a Kubeflow Pipeline (Module 11):

```
KFP Pipeline: credit-scoring-pipeline
  Step 1: ingest-data        (pull from data lake / connections)
  Step 2: feast-apply         (register feature definitions)
  Step 3: feast-materialize   (batch compute → online store)
  Step 4: train-model         (get_historical_features → XGBoost)
  Step 5: deploy-model        (KServe or custom serving)
  Step 6: validate-freshness  (check Prometheus metrics)
```

> **🗺️ DATA STRATEGY**: KFP is the orchestration backbone (Pillar 4). Feast steps integrate as pipeline components, not as operator-managed CRs.
>
> **⚠️ GAP**: No pre-built KFP component for Feast materialization with Ray/Spark — must build custom.

In [ ]:
# Pseudocode for KFP integration
kfp_pipeline = '''
from kfp import dsl

@dsl.component(base_image="quay.io/feastdev/feature-server:latest")
def feast_materialize(start_date: str, end_date: str):
    from feast import FeatureStore
    store = FeatureStore(repo_path="/opt/feast/feature_repo")
    store.materialize(
        datetime.fromisoformat(start_date),
        datetime.fromisoformat(end_date)
    )

@dsl.component(base_image="python:3.11")
def train_credit_model():
    from feast import FeatureStore
    store = FeatureStore(repo_path="/opt/feast/feature_repo")
    # ... get_historical_features, train XGBoost, log to MLflow

@dsl.pipeline(name="credit-scoring")
def credit_scoring_pipeline():
    mat = feast_materialize(start_date="2024-01-01", end_date="2024-12-31")
    train = train_credit_model()
    train.after(mat)
'''
print(kfp_pipeline)

## Summary

| Step | Feast Capability | Status |
|------|-----------------|--------|
| Data sources | FileSource, batch ingestion | GA |
| Feature engineering | Feature Views, ODFVs, FeatureService | GA |
| Materialize | Batch to online store | GA (manual scheduling) |
| Train | get_historical_features (point-in-time) | GA |
| Serve | get_online_features (low-latency) | GA |
| Lineage | OpenLineage emission | GA upstream, overlay on RHOAI |
| Orchestration | KFP pipeline integration | Custom components needed |

This is the **canonical Feast workload** — predictive AI with governed, consistent features from training through serving.

**Next**: [Module 15: Knowledge Retrieval Scenario](../15-scenario-knowledge-retrieval/15-scenario-knowledge-retrieval.ipynb) — Feast + RAG for existing adopters.